# Bottom-Up and Top-Down Attention for Image Captioning and Visual Question Answering

**Authors:** Peter Anderson, Xiaodong He, Chris Buehler, Damien Teney, Mark Johnson, Stephen Gould, Lei Zhang
**Venue:** arXiv:1707.07998v3 (CVPR 2018)

# https://arxiv.org/pdf/1707.07998

---

## Abstract

This paper proposes a combined bottom-up and top-down visual attention mechanism for image captioning and visual question answering (VQA). Bottom-up attention is implemented via Faster R-CNN to produce object-level region proposals, while top-down attention uses task-specific context to compute weighted combinations of those features. The approach achieves state-of-the-art performance on MSCOCO captioning and first place in the 2017 VQA Challenge.

---

## Problems

- Conventional visual attention in image captioning and VQA operates over a **uniform grid of equally-sized CNN spatial features**, irrespective of image content.
- Grid-based features impose an unresolvable trade-off between coarse and fine levels of detail.
- Arbitrary region positioning makes it difficult to detect and bind visual concepts associated with the same object (the **feature binding problem**).
- There is no principled connection between the regions subjected to attention and semantically meaningful image content (objects, attributes, relations).

---

## Proposed Solutions

- Introduce a **combined bottom-up and top-down attention mechanism**:
  - **Bottom-up attention**: Uses Faster R-CNN (with ResNet-101 backbone, pretrained on Visual Genome) to propose salient, object-aligned bounding box regions, each represented by a 2048-dimensional mean-pooled convolutional feature vector.
  - **Top-down attention**: Task-specific context (partial caption or question) computes a soft attention distribution over the proposed regions.
- Apply this mechanism to two tasks: **image captioning** (dual-LSTM architecture) and **VQA** (joint multimodal embedding with GRU question encoder).

---

## Purpose

- To provide a more natural and semantically grounded basis for visual attention by anchoring it at the level of objects and salient image regions rather than arbitrary spatial grid cells.
- To bridge vision-language tasks with advances in object detection, enabling cross-domain transfer through pretraining on detection datasets.

---

## Methodology

### Bottom-Up Attention Model

- Faster R-CNN with ResNet-101 backbone, pretrained on ImageNet, then fine-tuned on **Visual Genome** (108K images, 1,600 object classes, 400 attribute classes).
- Region selection: IoU threshold of 0.7 (RPN suppression), 0.3 (object suppression), confidence threshold of 0.2; top-$k$ regions selected per image (default $k = 36$).
- Each region $i$ produces feature $v_i \in \mathbb{R}^{2048}$.

### Captioning Model (Up-Down)

Two-layer LSTM architecture:

**Layer 1 — Top-Down Attention LSTM:**

$$x^1_t = [h^2_{t-1},\ \bar{v},\ W_e \Pi_t]$$

$$a_{i,t} = w_a^T \tanh(W_{va} v_i + W_{ha} h^1_t)$$

$$\alpha_t = \text{softmax}(a_t), \quad \hat{v}_t = \sum_{i=1}^{k} \alpha_{i,t} v_i$$

**Layer 2 — Language LSTM:**

$$x^2_t = [\hat{v}_t,\ h^1_t]$$

$$p(y_t \mid y_{1:t-1}) = \text{softmax}(W_p h^2_t + b_p)$$

Training uses cross-entropy loss followed by CIDEr optimization via **Self-Critical Sequence Training (SCST)**:

$$\nabla_\theta \mathcal{L}_R(\theta) \approx -(r(y^s_{1:T}) - r(\hat{y}_{1:T})) \nabla_\theta \log p_\theta(y^s_{1:T})$$

### VQA Model

- Questions encoded with a **GRU** (300-dim GloVe embeddings, 512-dim hidden state).
- Attention over image features using question context:

$$a_i = w_a^T f_a([v_i, q])$$

- Joint embedding via element-wise product of gated tanh activations:

$$h = f_q(q) \circ f_v(\hat{v}), \quad p(y) = \sigma(W_o f_o(h))$$

- Output: multi-label classifier over 3,129 candidate answers.

### Datasets

| Dataset | Usage |
|---|---|
| Visual Genome (108K images) | Pretraining bottom-up attention model |
| MSCOCO 2014 (113K train, 5K val/test) | Image captioning evaluation |
| VQA v2.0 (1.1M questions) | VQA evaluation and 2017 Challenge |

### Evaluation Metrics

- Captioning: BLEU-4, METEOR, ROUGE-L, CIDEr, SPICE
- VQA: Standard VQA accuracy (Yes/No, Number, Other, Overall)

---

## Results

### Image Captioning (MSCOCO Karpathy Test Split — Single Model)

| Model | BLEU-4 | METEOR | ROUGE-L | CIDEr | SPICE |
|---|---|---|---|---|---|
| SCST:Att2all | 34.2 | 26.7 | 55.7 | 114.0 | — |
| Ours: ResNet | 34.0 | 26.5 | 54.9 | 111.1 | 20.2 |
| **Ours: Up-Down** | **36.3** | **27.7** | **56.9** | **120.1** | **21.4** |

Relative improvement of **3–8%** over the ResNet baseline across all metrics.

### Image Captioning (MSCOCO Test Server — 4-Model Ensemble)

$$\text{CIDEr} = 117.9,\quad \text{SPICE} = 21.5,\quad \text{BLEU-4} = 36.9$$

State-of-the-art at the time of submission (July 2017), outperforming all published and unpublished entries.

### VQA v2.0 (Single Model — Validation Set)

| Model | Yes/No | Number | Other | Overall |
|---|---|---|---|---|
| ResNet (7×7) | 77.6 | 37.7 | 51.5 | 59.4 |
| **Up-Down** | **80.3** | **42.8** | **55.8** | **63.2** |

### VQA v2.0 (30-Model Ensemble — Test-Standard Server)

$$\text{Overall Accuracy} = 70.34\%$$

First place in the **2017 VQA Challenge**, outperforming all other leaderboard entries.

---

## Conclusions

- Object-level bottom-up attention provides a significantly more natural and effective basis for visual attention than uniform CNN grid features, yielding consistent 3–8% gains across all captioning metrics and substantial VQA improvements.
- The approach bridges image captioning and VQA with recent progress in object detection, allowing cross-domain knowledge transfer via pretraining on Visual Genome.
- The attention mechanism improves interpretability: attended regions correspond to semantically coherent objects rather than arbitrary spatial patches.
- The immediate practical benefit is straightforward: pretrained bottom-up attention features can serve as a drop-in replacement for CNN features in existing vision-language models.
- Identified limitations include reduced performance on tasks requiring fine-grained reading (OCR) and precise counting, reflecting the bounds of the object-proposal paradigm.

# Mathematical and Statistical Content: Bottom-Up and Top-Down Attention

---

## 1. Image Feature Representation

$$V = \{v_1, \dots, v_k\}, \quad v_i \in \mathbb{R}^{D},\quad D = 2048$$

The image is represented as a set of $k$ region feature vectors. Each $v_i$ is a
2048-dimensional vector extracted from a bounding box region via mean-pooled
convolutional features from ResNet-101. This replaces the conventional fixed
spatial grid with a variable-length, object-aligned set of features.

---

## 2. Mean-Pooled Image Feature

$$\bar{v} = \frac{1}{k} \sum_{i=1}^{k} v_i$$

A simple arithmetic mean over all $k$ region features. It serves as a global
image summary, providing the attention LSTM with an overall sense of image
content at every time step, independent of what word is currently being
generated.

---

## 3. LSTM Recurrence (Notation)

$$h_t = \text{LSTM}(x_t,\ h_{t-1})$$

Standard LSTM update: given input $x_t$ and previous hidden state $h_{t-1}$,
produce new hidden state $h_t$. The paper uses two stacked LSTM layers — one
for attention and one for language generation — each following this recurrence.
Memory cell propagation is suppressed in the notation for brevity.

---

## 4. Attention LSTM Input Vector

$$x^1_t = [h^2_{t-1},\ \bar{v},\ W_e \Pi_t]$$

The input to the first (attention) LSTM at time step $t$ is a concatenation of
three components:

| Component | Meaning |
|---|---|
| $h^2_{t-1}$ | Previous hidden state of the language LSTM |
| $\bar{v}$ | Global mean image feature |
| $W_e \Pi_t$ | Word embedding of the previously generated word |

$W_e \in \mathbb{R}^{E \times |\Sigma|}$ is a learned embedding matrix,
$\Pi_t$ is a one-hot encoding of the input word at time $t$, and $\Sigma$ is
the vocabulary. This gives the attention LSTM full context: language state,
image content, and caption history.

---

## 5. Unnormalized Attention Score

$$a_{i,t} = w_a^T \tanh\!\left(W_{va}\, v_i + W_{ha}\, h^1_t\right)$$

For each image region $i$ at time step $t$, a scalar attention score is
computed. The region feature $v_i$ and the attention LSTM hidden state
$h^1_t$ are linearly projected into a shared space, passed through $\tanh$
(element-wise nonlinearity), then projected to a scalar by $w_a$.

**Parameters:**
$$W_{va} \in \mathbb{R}^{H \times V},\quad W_{ha} \in \mathbb{R}^{H \times M},\quad w_a \in \mathbb{R}^H$$

where $H$ is the attention hidden size, $V = 2048$ is the feature dimension,
and $M$ is the LSTM hidden size.

---

## 6. Softmax Normalization of Attention

$$\alpha_t = \text{softmax}(a_t), \quad \text{i.e.,}\quad \alpha_{i,t} = \frac{\exp(a_{i,t})}{\sum_{j=1}^{k} \exp(a_{j,t})}$$

Converts the raw attention scores into a valid probability distribution over
the $k$ regions. Each $\alpha_{i,t} \in (0,1)$ and
$\sum_i \alpha_{i,t} = 1$. This is a **soft attention** mechanism — all
regions contribute, weighted by their relevance.

---

## 7. Attended Image Feature (Convex Combination)

$$\hat{v}_t = \sum_{i=1}^{k} \alpha_{i,t}\, v_i$$

The attended feature is a weighted sum of all region features, where the
weights come from the softmax attention distribution. Because the weights sum
to one and are positive, this is a **convex combination** — the result lives
in the convex hull of the region features. It produces a single
context-sensitive image representation for each decoding time step.

---

## 8. Language LSTM Input Vector

$$x^2_t = [\hat{v}_t,\ h^1_t]$$

The second (language) LSTM receives the attended image feature concatenated
with the attention LSTM's output. This couples visual context directly into
the word generation process at every step.

---

## 9. Output Word Distribution

$$p(y_t \mid y_{1:t-1}) = \text{softmax}\!\left(W_p\, h^2_t + b_p\right)$$

A linear projection of the language LSTM's hidden state, followed by softmax,
produces a probability distribution over the vocabulary $\Sigma$ at each time
step. $W_p \in \mathbb{R}^{|\Sigma| \times M}$, $b_p \in \mathbb{R}^{|\Sigma|}$.

---

## 10. Joint Sequence Probability

$$p(y_{1:T}) = \prod_{t=1}^{T} p(y_t \mid y_{1:t-1})$$

The probability of a complete caption of length $T$ is the product of
per-step conditional probabilities — the standard autoregressive factorization
for sequence generation. The model is trained to maximize this quantity on
ground-truth captions.

---

## 11. Cross-Entropy Training Loss

$$\mathcal{L}_{XE}(\theta) = -\sum_{t=1}^{T} \log\!\left(p_\theta(y^*_t \mid y^*_{1:t-1})\right)$$

Standard supervised training objective. For each time step, the negative
log-probability of the ground-truth word $y^*_t$ is summed. Minimizing this
encourages the model to assign high probability to the correct next word given
the correct previous words (teacher forcing).

---

## 12. Reinforcement Learning / CIDEr Optimization Loss

$$\mathcal{L}_R(\theta) = -\mathbb{E}_{y_{1:T} \sim p_\theta}\!\left[r(y_{1:T})\right]$$

Instead of maximizing word-level log-likelihood, this objective maximizes the
**expected sequence-level reward** $r$ (CIDEr score). Because CIDEr is
non-differentiable, the expectation is approximated via the REINFORCE policy
gradient estimator (Self-Critical Sequence Training, SCST):

$$\nabla_\theta \mathcal{L}_R(\theta) \approx -\!\left(r(y^s_{1:T}) - r(\hat{y}_{1:T})\right) \nabla_\theta \log p_\theta(y^s_{1:T})$$

| Symbol | Meaning |
|---|---|
| $y^s_{1:T}$ | Caption sampled from the model during training |
| $\hat{y}_{1:T}$ | Greedy-decoded (baseline) caption from the current model |
| $r(\cdot)$ | CIDEr score function |

The term $r(y^s_{1:T}) - r(\hat{y}_{1:T})$ is the **advantage**: sampled
captions that score higher than the greedy baseline receive a positive
gradient update, reinforcing their generation. This eliminates exposure bias
from teacher forcing and directly optimizes the evaluation metric.

---

## 13. Gated Tanh Activation (VQA Model)

$$\tilde{y} = \tanh(Wx + b)$$
$$g = \sigma(W'x + b')$$
$$y = \tilde{y} \circ g$$

A **gated hyperbolic tangent** unit, a special case of highway networks.
The sigmoid gate $g \in (0,1)^n$ controls how much of the tanh activation
$\tilde{y}$ passes through element-wise ($\circ$ denotes the Hadamard
product). This provides richer non-linear transformations than standard ReLU
or tanh layers, with empirically stronger performance on VQA.

**Parameters:** $W, W' \in \mathbb{R}^{n \times m}$, $b, b' \in \mathbb{R}^n$.

---

## 14. VQA Attention Score

$$a_i = w_a^T f_a([v_i,\ q])$$

Analogous to the captioning attention score (Equation 5), but uses the
question encoding $q$ (GRU hidden state) instead of the LSTM hidden state.
The concatenation $[v_i, q]$ is passed through a gated tanh layer $f_a$,
then projected to a scalar. The same softmax normalization (Equation 6) and
convex combination (Equation 7) follow to produce $\hat{v}$.

---

## 15. VQA Joint Embedding and Output

$$h = f_q(q) \circ f_v(\hat{v})$$
$$p(y) = \sigma\!\left(W_o\, f_o(h)\right)$$

The question representation $q$ and attended image feature $\hat{v}$ are
each passed through separate gated tanh layers, then combined via element-wise
product to form a joint multimodal embedding $h$. A final gated tanh layer
$f_o$ followed by a sigmoid (not softmax) produces independent scores for
each of the 3,129 candidate answers — framing VQA as **multi-label
classification** rather than single-answer selection, accommodating annotator
disagreement in the ground truth.

---

## 16. Summary Table of Key Mathematical Operations

| Operation | Mathematical Form | Role |
|---|---|---|
| Feature set | $V = \{v_i\} \in \mathbb{R}^{k \times 2048}$ | Object-level image representation |
| Global image summary | $\bar{v} = \frac{1}{k}\sum v_i$ | Context for attention LSTM |
| Attention score | $a_{i,t} = w_a^T \tanh(W_{va}v_i + W_{ha}h^1_t)$ | Region relevance scoring |
| Attention weight | $\alpha_t = \text{softmax}(a_t)$ | Normalised distribution over regions |
| Attended feature | $\hat{v}_t = \sum \alpha_{i,t} v_i$ | Weighted visual context vector |
| Caption probability | $p(y_t \mid y_{1:t-1}) = \text{softmax}(W_p h^2_t + b_p)$ | Word generation |
| XE loss | $\mathcal{L}_{XE} = -\sum \log p_\theta(y^*_t)$ | Supervised sequence training |
| RL loss gradient | $-(r(y^s) - r(\hat{y}))\nabla \log p_\theta(y^s)$ | CIDEr-optimised fine-tuning |
| Gated tanh | $\tilde{y} \circ \sigma(W'x + b')$ | Non-linear VQA transformation |
| VQA joint embedding | $h = f_q(q) \circ f_v(\hat{v})$ | Multimodal fusion |

# Problem–Gap–Solution Analysis
## Bottom-Up and Top-Down Attention for Image Captioning and Visual Question Answering

---

| # | Problem / Research Gap | Limitation of Prior Work | Proposed Solution |
|---|---|---|---|
| 1 | Visual attention in image captioning and VQA operates over a **uniform grid of equally-sized CNN spatial regions**, regardless of image content | Grid-based features are spatially arbitrary and semantically uninformed; the attended regions bear no inherent relationship to objects or meaningful scene components | Replace grid features with object-aligned region proposals from **Faster R-CNN** (bottom-up attention), ensuring each attended unit corresponds to a salient, semantically coherent image region |
| 2 | There is an inherent **trade-off between coarse and fine levels of spatial detail** in fixed-size CNN feature maps | Selecting a single spatial resolution forces the model to either miss fine-grained detail or lose broader contextual information; no single grid size resolves both simultaneously | Faster R-CNN generates region proposals of **varying scales and aspect ratios**, allowing the model to naturally attend to both small, fine-grained details and large contextual regions without resolution trade-offs |
| 3 | Conventional attention mechanisms struggle with the **feature binding problem**: visual attributes of the same object are spatially dispersed across multiple grid cells | Grid-based regions may split an object across multiple cells, making it difficult to associate all visual properties (shape, color, texture, relation) of a single object | Object-level bounding box proposals ensure that **all visual concepts associated with a single object are spatially co-located** and processed together as a unified attended unit |
| 4 | Prior attention models are exclusively **top-down**, driven only by task context (partial caption or question), with no bottom-up visual saliency signal | The model has no principled mechanism to identify which image regions are inherently salient or visually distinctive independent of the task; attention is entirely conditioned on linguistic context | Combine **bottom-up saliency** (object detection priors from Visual Genome pretraining) with **top-down task context** (LSTM or GRU hidden states) to compute attention that reflects both visual and linguistic relevance |
| 5 | Region proposals in prior captioning work relied on **hand-crafted methods** (selective search, edge boxes) or differentiable but weakly supervised spatial transformers | Hand-crafted proposals do not leverage large-scale object detection supervision; they lack semantic grounding and cannot transfer cross-domain knowledge from detection datasets | Leverage **Faster R-CNN pretrained on Visual Genome** with 1,600 object classes and 400 attribute classes, establishing a principled link to object detection and enabling large-scale cross-domain pretraining analogous to ImageNet transfer |
| 6 | Standard sequence training with **cross-entropy loss suffers from exposure bias**: the model is trained on ground-truth tokens but evaluated on its own generated sequences | Discrepancy between training and inference conditions leads to error accumulation; furthermore, cross-entropy does not directly optimize sequence-level evaluation metrics such as CIDEr | Fine-tune the captioning model using **Self-Critical Sequence Training (SCST)**, a REINFORCE-based policy gradient method that directly optimizes CIDEr by using the greedy-decoded caption score as a self-generated baseline |
| 7 | VQA models based on uniform CNN features show **limited sensitivity to visually grounded object-level evidence**, particularly for questions requiring identification of specific objects or attributes | Flat spatial features conflate object and background signals; the model cannot reliably isolate the visual evidence most relevant to a given question | Apply the same bottom-up attention features to the VQA model, enabling the **question-conditioned top-down attention to select among pre-identified object regions** rather than arbitrary spatial locations, improving grounding and answer accuracy |
| 8 | Prior top-down attention models offer **limited interpretability**: attended spatial grid cells do not correspond to human-recognizable entities | Attention weight visualizations over grid cells are difficult to interpret semantically, reducing the diagnostic and explanatory value of the attention mechanism | Object-aligned attention regions produce **directly interpretable attention maps** where each attended region corresponds to a named object or salient patch, as demonstrated qualitatively in Figures 5–10 |

---

## Summary of Core Contribution

The unifying contribution of the paper is the insight that **the choice of what image regions are available for attention is as important as how attention weights are computed**. Prior work treated region definition as a fixed preprocessing step (uniform grid), while this paper treats it as a **learnable, semantically grounded module** (Faster R-CNN with detection pretraining), fundamentally improving both task performance and attention interpretability across image captioning and VQA.

In [ ]:
# ============================================================
# Bottom-Up and Top-Down Attention for Image Captioning
# Full Educational PyTorch Implementation
# Based on: Anderson et al., CVPR 2018
# ============================================================
# SETUP (Google Colab):
# !pip install torch torchvision transformers datasets
# !pip install pycocotools nltk rouge-score
# Runtime: GPU (T4 or better recommended)
# ============================================================

# ============================================================
# SECTION 0: IMPORTS AND CONFIGURATION
# ============================================================

import os
import json
import math
import time
import random
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet101, ResNet101_Weights

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Hyperparameters ──────────────────────────────────────────
CFG = {
    # Data
    'max_caption_len'   : 20,
    'min_word_freq'     : 2,
    'num_regions'       : 36,        # fixed bottom-up regions per image
    'region_feat_dim'   : 2048,      # ResNet-101 pool5 output

    # Model
    'embed_dim'         : 512,
    'attention_dim'     : 512,
    'lstm_hidden_dim'   : 512,
    'dropout'           : 0.5,

    # Training
    'batch_size'        : 32,
    'num_epochs_xe'     : 5,         # cross-entropy phase
    'num_epochs_rl'     : 3,         # SCST / RL fine-tuning phase
    'lr_xe'             : 4e-4,
    'lr_rl'             : 4e-5,
    'grad_clip'         : 5.0,
    'beam_size'         : 3,

    # CIFAR-10 proxy (replaces MSCOCO for self-contained demo)
    'num_train_samples' : 2000,
    'num_val_samples'   : 400,
    'num_test_samples'  : 200,
    'img_size'          : 32,
}

# ── Output directory ─────────────────────────────────────────
OUT = Path('updown_outputs')
OUT.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"  Region features : {CFG['num_regions']} x {CFG['region_feat_dim']}")
print(f"  Embed dim       : {CFG['embed_dim']}")
print(f"  LSTM hidden     : {CFG['lstm_hidden_dim']}")


# ============================================================
# SECTION 1: DATASET — CIFAR-10 WITH SYNTHETIC CAPTIONS
# ============================================================
# Educational proxy for MSCOCO:
#   • Each CIFAR-10 image has 5 synthetic ground-truth captions.
#   • Region features are simulated from a CNN backbone.
#   • This allows us to demonstrate every architectural component
#     without requiring a 13 GB MSCOCO download.
# ============================================================

CIFAR10_LABELS = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Caption templates for each class (5 per class)
CAPTION_TEMPLATES = {
    'airplane' : [
        "a small airplane flying through the blue sky",
        "an airplane soaring high above the clouds",
        "a silver airplane with wings spread wide",
        "a jet airplane moving quickly across the sky",
        "a white airplane flying in clear weather"
    ],
    'automobile': [
        "a red automobile parked on the street",
        "a fast automobile driving down the road",
        "a shiny automobile with four wheels",
        "a small automobile seen from above",
        "a colorful automobile moving through traffic"
    ],
    'bird'      : [
        "a small bird perched on a branch",
        "a colorful bird with bright feathers",
        "a bird flying freely in the open sky",
        "a tiny bird sitting on a green tree",
        "a bird with spread wings in flight"
    ],
    'cat'       : [
        "a fluffy cat sitting on a soft surface",
        "a cute cat with bright eyes looking ahead",
        "a small cat resting on a warm mat",
        "a striped cat playing with a toy",
        "a white and black cat curled up asleep"
    ],
    'deer'      : [
        "a brown deer standing in the green forest",
        "a deer with large antlers in the wild",
        "a graceful deer running through tall grass",
        "a young deer grazing in a meadow",
        "a deer pausing and looking at the camera"
    ],
    'dog'       : [
        "a happy dog running across the green lawn",
        "a furry dog sitting and looking up",
        "a playful dog with a wagging tail",
        "a brown dog lying on the wooden floor",
        "a cute dog with floppy ears and bright eyes"
    ],
    'frog'      : [
        "a green frog sitting on a wet leaf",
        "a small frog near the edge of a pond",
        "a slimy frog resting on a mossy rock",
        "a frog with wide eyes looking forward",
        "a tiny frog camouflaged in green grass"
    ],
    'horse'     : [
        "a brown horse galloping across an open field",
        "a tall horse standing beside a wooden fence",
        "a strong horse with a flowing mane",
        "a black horse running freely in the meadow",
        "a horse grazing quietly on green grass"
    ],
    'ship'      : [
        "a large ship sailing across the blue ocean",
        "a white ship moving through calm waters",
        "a cargo ship floating near the harbor",
        "a big ship with tall masts on the sea",
        "a sailing ship traveling under a cloudy sky"
    ],
    'truck'     : [
        "a large red truck driving on the highway",
        "a heavy truck loaded with cargo",
        "a big blue truck parked at the loading dock",
        "a truck with bright headlights on the road",
        "a delivery truck moving through the city street"
    ],
}


class Vocabulary:
    """
    Vocabulary: maps words <-> integer indices.

    Special tokens:
      <pad>  index 0  — padding to equalize sequence lengths
      <start> index 1 — signals the start of a caption
      <end>  index 2  — signals the end of a caption
      <unk>  index 3  — unknown / rare word
    """
    PAD, START, END, UNK = 0, 1, 2, 3

    def __init__(self):
        self.word2idx = {'<pad>': 0, '<start>': 1, '<end>': 2, '<unk>': 3}
        self.idx2word = {0: '<pad>', 1: '<start>', 2: '<end>', 3: '<unk>'}
        self.freq     = Counter()

    def build(self, captions, min_freq=2):
        for cap in captions:
            for w in cap.lower().split():
                self.freq[w] += 1
        for w, f in self.freq.items():
            if f >= min_freq and w not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[w] = idx
                self.idx2word[idx] = w
        print(f"Vocabulary built: {len(self.word2idx)} words "
              f"(min_freq={min_freq})")

    def encode(self, caption, max_len):
        """Convert caption string to padded index tensor."""
        tokens = ['<start>'] + caption.lower().split() + ['<end>']
        tokens = tokens[:max_len]
        ids = [self.word2idx.get(t, self.UNK) for t in tokens]
        # Pad to max_len
        ids += [self.PAD] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        """Convert index list back to a caption string."""
        words = []
        for i in ids:
            w = self.idx2word.get(i, '<unk>')
            if w == '<end>':
                break
            if w not in ('<pad>', '<start>'):
                words.append(w)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)


def build_vocab_and_data():
    """
    Collect all caption templates, build vocabulary,
    and return the caption list for dataset construction.
    """
    all_captions = []
    for caps in CAPTION_TEMPLATES.values():
        all_captions.extend(caps)

    vocab = Vocabulary()
    vocab.build(all_captions, min_freq=CFG['min_word_freq'])
    return vocab, all_captions


class CIFAR10CaptionDataset(Dataset):
    """
    CIFAR-10 + Synthetic Captions dataset.

    Each item returns:
      regions  : (num_regions, region_feat_dim) — simulated bottom-up features
      caption  : (max_caption_len,)             — encoded word indices
      cap_len  : int                            — true caption length
      label    : int                            — class index (for reference)
    """

    def __init__(self, split, vocab, num_samples):
        super().__init__()
        self.vocab       = vocab
        self.num_regions = CFG['num_regions']
        self.feat_dim    = CFG['region_feat_dim']
        self.max_len     = CFG['max_caption_len']

        # Download / load CIFAR-10
        transform = transforms.Compose([
            transforms.Resize((CFG['img_size'], CFG['img_size'])),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_flag = (split in ['train', 'val'])
        base = torchvision.datasets.CIFAR10(
            root='./data', train=train_flag,
            download=True, transform=transform
        )

        # Sub-sample for speed
        indices = list(range(min(num_samples, len(base))))
        random.shuffle(indices)
        self.images = [base[i][0] for i in indices]
        self.labels = [base[i][1] for i in indices]

        # Assign one ground-truth caption per image
        # (in real MSCOCO every image has 5 captions)
        self.captions = []
        for lbl in self.labels:
            cls_name = CIFAR10_LABELS[lbl]
            cap = random.choice(CAPTION_TEMPLATES[cls_name])
            self.captions.append(cap)

        # Pre-compute simple CNN backbone for region features
        self._backbone = self._build_backbone()

    def _build_backbone(self):
        """
        Lightweight feature extractor to simulate Faster R-CNN
        region features. We use ResNet-101 avgpool output as a
        global feature and tile it for all regions with noise.
        """
        model = resnet101(weights=ResNet101_Weights.DEFAULT)
        # Remove final classifier
        backbone = nn.Sequential(*list(model.children())[:-1])
        backbone.eval()
        return backbone

    def _extract_region_features(self, img_tensor):
        """
        Simulate k=36 region features.
        In production: Faster R-CNN extracts one 2048-d vector
        per detected bounding box. Here we tile the global
        pool5 feature with additive Gaussian noise to create
        region diversity while keeping training feasible.
        """
        with torch.no_grad():
            # img_tensor: (3, H, W) — add batch dim
            feat = self._backbone(img_tensor.unsqueeze(0))  # (1,2048,1,1)
            feat = feat.squeeze()                           # (2048,)
            # Tile + noise → (num_regions, 2048)
            regions = feat.unsqueeze(0).repeat(self.num_regions, 1)
            noise   = torch.randn_like(regions) * 0.1
            regions = regions + noise
        return regions  # (36, 2048)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img      = self.images[idx]
        cap_str  = self.captions[idx]
        label    = self.labels[idx]

        # Region features: (num_regions, region_feat_dim)
        regions  = self._extract_region_features(img)

        # Encoded caption: (max_caption_len,)
        cap_ids  = self.vocab.encode(cap_str, self.max_len)
        cap_ids  = torch.tensor(cap_ids, dtype=torch.long)

        # True caption length (capped at max_len)
        words    = cap_str.lower().split()
        cap_len  = min(len(words) + 2, self.max_len)  # +2 for <start><end>

        return regions, cap_ids, cap_len, label


def build_datasets(vocab):
    print("Building datasets...")
    train_ds = CIFAR10CaptionDataset('train', vocab, CFG['num_train_samples'])
    val_ds   = CIFAR10CaptionDataset('val',   vocab, CFG['num_val_samples'])
    test_ds  = CIFAR10CaptionDataset('test',  vocab, CFG['num_test_samples'])
    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)} | "
          f"Test: {len(test_ds)}")
    return train_ds, val_ds, test_ds


def build_loaders(train_ds, val_ds, test_ds):
    train_loader = DataLoader(
        train_ds, batch_size=CFG['batch_size'],
        shuffle=True,  num_workers=0, pin_memory=False
    )
    val_loader = DataLoader(
        val_ds,   batch_size=CFG['batch_size'],
        shuffle=False, num_workers=0, pin_memory=False
    )
    test_loader = DataLoader(
        test_ds,  batch_size=CFG['batch_size'],
        shuffle=False, num_workers=0, pin_memory=False
    )
    return train_loader, val_loader, test_loader


# ============================================================
# SECTION 2: MODEL ARCHITECTURE
# ============================================================
# The Up-Down model has three main components:
#
#  A. Attention Module       — scores each region against LSTM state
#  B. Top-Down Attention LSTM — attends to image regions at each step
#  C. Language LSTM           — generates next word from attended feature
#
# ============================================================

class AttentionModule(nn.Module):
    """
    Soft Top-Down Attention (Equation 3–5 in the paper).

    At each caption time step, given:
      - The attention LSTM hidden state  h1  : (B, M)
      - All region features              V   : (B, k, D)

    Computes a normalized attention distribution α over the k regions,
    then returns the attended feature v̂ as a convex combination.

    Equations:
        a_i = w_a^T tanh( W_va v_i  +  W_ha h1 )      [score each region]
        α   = softmax(a)                                [normalize]
        v̂   = Σ_i α_i v_i                              [weighted sum]
    """

    def __init__(self, region_dim, lstm_dim, attention_dim):
        super().__init__()
        # Projects region features into attention space
        self.W_va = nn.Linear(region_dim, attention_dim, bias=False)
        # Projects LSTM hidden state into attention space
        self.W_ha = nn.Linear(lstm_dim,   attention_dim, bias=False)
        # Final projection to scalar score
        self.w_a  = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, V, h1):
        """
        Args:
            V  : (B, k, region_dim)  — region feature set
            h1 : (B, lstm_dim)       — attention LSTM hidden state
        Returns:
            v_hat : (B, region_dim)  — attended image feature
            alpha : (B, k)           — attention weights
        """
        B, k, _ = V.size()

        # Project regions: (B, k, attention_dim)
        proj_V = self.W_va(V)

        # Project h1 and broadcast over k regions: (B, k, attention_dim)
        proj_h = self.W_ha(h1).unsqueeze(1).expand(B, k, -1)

        # Compute unnormalized attention scores: (B, k, 1) → (B, k)
        energy = self.w_a(torch.tanh(proj_V + proj_h)).squeeze(2)

        # Softmax normalization over k regions: (B, k)
        alpha = F.softmax(energy, dim=1)

        # Attended feature: (B, region_dim)
        # Σ_i α_i v_i  — convex combination of region vectors
        v_hat = (alpha.unsqueeze(2) * V).sum(dim=1)

        return v_hat, alpha


class UpDownCaptioner(nn.Module):
    """
    Up-Down Image Captioning Model (Section 3.2 of the paper).

    Architecture:
      ┌──────────────────────────────────────────┐
      │  Layer 1: Top-Down Attention LSTM         │
      │    input  = [h2_{t-1}, v̄, We Π_t]        │
      │    output = h1_t                          │
      ├──────────────────────────────────────────┤
      │  Attention Module                         │
      │    input  = (V, h1_t)                     │
      │    output = v̂_t, α_t                     │
      ├──────────────────────────────────────────┤
      │  Layer 2: Language LSTM                   │
      │    input  = [v̂_t, h1_t]                  │
      │    output = h2_t                          │
      ├──────────────────────────────────────────┤
      │  Classifier: softmax(W_p h2_t + b_p)     │
      └──────────────────────────────────────────┘
    """

    def __init__(self, vocab_size, region_dim, embed_dim,
                 lstm_dim, attention_dim, dropout=0.5):
        super().__init__()

        self.vocab_size    = vocab_size
        self.region_dim    = region_dim
        self.embed_dim     = embed_dim
        self.lstm_dim      = lstm_dim

        # Word embedding matrix  W_e : |Σ| → E
        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=0)

        # ── Layer 1: Top-Down Attention LSTM ─────────────────
        # Input dimension = lstm_dim (h2_{t-1})
        #                 + region_dim (v̄)
        #                 + embed_dim  (W_e Π_t)
        self.attention_lstm = nn.LSTMCell(
            input_size  = lstm_dim + region_dim + embed_dim,
            hidden_size = lstm_dim
        )

        # Soft attention module
        self.attention = AttentionModule(region_dim, lstm_dim, attention_dim)

        # ── Layer 2: Language LSTM ───────────────────────────
        # Input dimension = region_dim (v̂_t)
        #                 + lstm_dim   (h1_t)
        self.language_lstm = nn.LSTMCell(
            input_size  = region_dim + lstm_dim,
            hidden_size = lstm_dim
        )

        # Output classifier  W_p : M → |Σ|
        self.fc      = nn.Linear(lstm_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def _mean_pool(self, V):
        """
        Global image summary: v̄ = (1/k) Σ_i v_i
        Shape: (B, region_dim)
        """
        return V.mean(dim=1)

    def _init_hidden(self, B):
        """Zero-initialize both LSTM hidden and cell states."""
        h = torch.zeros(B, self.lstm_dim, device=DEVICE)
        c = torch.zeros(B, self.lstm_dim, device=DEVICE)
        return h, c

    def forward(self, V, captions, lengths):
        """
        Teacher-forced forward pass for cross-entropy training.

        Args:
            V        : (B, k, region_dim)  — region features
            captions : (B, max_len)        — encoded word indices
            lengths  : (B,)               — true caption lengths

        Returns:
            logits   : (B, max_len-1, vocab_size)
            alphas   : (B, max_len-1, k)
        """
        B       = V.size(0)
        v_bar   = self._mean_pool(V)           # (B, region_dim)

        # Initialize LSTM states
        h1, c1  = self._init_hidden(B)         # attention LSTM
        h2, c2  = self._init_hidden(B)         # language LSTM

        max_t   = max(lengths) - 1             # exclude <end>
        all_logits = []
        all_alphas = []

        for t in range(max_t):
            # ── Word embedding ────────────────────────────────
            # W_e Π_t : (B, embed_dim)
            w_emb = self.dropout(self.embedding(captions[:, t]))

            # ── Attention LSTM input ──────────────────────────
            # x1_t = [h2_{t-1}, v̄, W_e Π_t]
            x1 = torch.cat([h2, v_bar, w_emb], dim=1)
            h1, c1 = self.attention_lstm(x1, (h1, c1))

            # ── Soft Attention ───────────────────────────────
            # v̂_t, α_t = Attend(V, h1_t)
            v_hat, alpha = self.attention(V, h1)  # (B,D), (B,k)

            # ── Language LSTM input ───────────────────────────
            # x2_t = [v̂_t, h1_t]
            x2 = torch.cat([v_hat, h1], dim=1)
            h2, c2 = self.language_lstm(x2, (h2, c2))

            # ── Word prediction ───────────────────────────────
            # p(y_t) = softmax(W_p h2_t + b_p)
            logit = self.fc(self.dropout(h2))     # (B, vocab_size)
            all_logits.append(logit)
            all_alphas.append(alpha)

        # Stack along time dimension
        logits = torch.stack(all_logits, dim=1)   # (B, T-1, vocab_size)
        alphas = torch.stack(all_alphas, dim=1)   # (B, T-1, k)

        return logits, alphas

    def generate_greedy(self, V, vocab, max_len=20):
        """
        Greedy decoding: at each step pick the highest-probability word.
        Used as the baseline in SCST and for evaluation.
        """
        B     = V.size(0)
        v_bar = self._mean_pool(V)
        h1, c1 = self._init_hidden(B)
        h2, c2 = self._init_hidden(B)

        word = torch.full((B,), vocab.START,
                          dtype=torch.long, device=DEVICE)
        captions = [[] for _ in range(B)]
        all_alphas = []

        for _ in range(max_len):
            w_emb = self.embedding(word)
            x1    = torch.cat([h2, v_bar, w_emb], dim=1)
            h1, c1 = self.attention_lstm(x1, (h1, c1))
            v_hat, alpha = self.attention(V, h1)
            x2    = torch.cat([v_hat, h1], dim=1)
            h2, c2 = self.language_lstm(x2, (h2, c2))
            logit  = self.fc(h2)
            word   = logit.argmax(dim=1)
            all_alphas.append(alpha.detach().cpu())
            for b in range(B):
                if word[b].item() != vocab.END:
                    captions[b].append(word[b].item())

        stacked_alphas = torch.stack(all_alphas, dim=1)  # (B, T, k)
        return captions, stacked_alphas

    def generate_beam(self, V, vocab, beam_size=3, max_len=20):
        """
        Beam search decoding.
        Maintains beam_size candidate sequences at each step,
        expanding and pruning to keep the top-scoring hypotheses.
        Returns the single best caption for the first image in V.
        """
        # Process single image only (B=1)
        v_bar  = self._mean_pool(V)            # (1, D)
        h1, c1 = self._init_hidden(1)
        h2, c2 = self._init_hidden(1)

        # Each beam entry: (score, token_list, h1, c1, h2, c2)
        beams = [(0.0, [vocab.START], h1, c1, h2, c2)]
        completed = []

        for _ in range(max_len):
            new_beams = []
            for score, tokens, h1b, c1b, h2b, c2b in beams:
                if tokens[-1] == vocab.END:
                    completed.append((score, tokens))
                    continue
                word  = torch.tensor([tokens[-1]], device=DEVICE)
                w_emb = self.embedding(word)
                x1    = torch.cat([h2b, v_bar, w_emb], dim=1)
                h1n, c1n = self.attention_lstm(x1, (h1b, c1b))
                v_hat, _ = self.attention(V, h1n)
                x2    = torch.cat([v_hat, h1n], dim=1)
                h2n, c2n = self.language_lstm(x2, (h2n := h1n, c2b))
                # Correction: use h2n properly
                x2    = torch.cat([v_hat, h1n], dim=1)
                h2n, c2n = self.language_lstm(x2, (h2b, c2b))
                log_probs = F.log_softmax(self.fc(h2n), dim=1)
                topk_lp, topk_idx = log_probs.topk(beam_size)
                for k in range(beam_size):
                    new_beams.append((
                        score + topk_lp[0, k].item(),
                        tokens + [topk_idx[0, k].item()],
                        h1n, c1n, h2n, c2n
                    ))
            beams = sorted(new_beams, key=lambda x: x[0],
                           reverse=True)[:beam_size]
            if not beams:
                break

        completed += [(s, t) for s, t, *_ in beams]
        best = max(completed, key=lambda x: x[0])
        return [w for w in best[1][1:] if w != vocab.END]


# ============================================================
# SECTION 3: LOSS FUNCTIONS
# ============================================================

class CrossEntropyLoss(nn.Module):
    """
    Standard cross-entropy sequence loss (Equation 9 in the paper).

        L_XE(θ) = - Σ_t log p_θ(y*_t | y*_{1:t-1})

    Ignores padding positions using a mask derived from lengths.
    """

    def __init__(self, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx

    def forward(self, logits, targets, lengths):
        """
        Args:
            logits  : (B, T-1, vocab_size)
            targets : (B, T)   — includes <end> token
            lengths : (B,)

        Returns:
            scalar loss averaged over non-pad tokens
        """
        B, T_1, V = logits.size()

        # Targets are words at positions 1..T (after <start>)
        tgt = targets[:, 1:1 + T_1]            # (B, T-1)

        # Build mask: 1 where token is real, 0 where padding
        mask = torch.zeros_like(tgt, dtype=torch.bool)
        for b in range(B):
            mask[b, :min(lengths[b] - 1, T_1)] = True

        # Flatten for F.cross_entropy
        logits_flat = logits.reshape(-1, V)
        tgt_flat    = tgt.reshape(-1)
        mask_flat   = mask.reshape(-1)

        loss_all = F.cross_entropy(logits_flat, tgt_flat,
                                   reduction='none')
        loss = loss_all[mask_flat].mean()
        return loss


def simple_cider_score(hypothesis, reference):
    """
    Lightweight CIDEr approximation using n-gram overlap (n=1..4).
    Used as the reward signal in SCST training.

    In production use the pycocoevalcap CIDEr implementation.
    Here we compute a simplified cosine-similarity-based n-gram score.
    """

    def get_ngrams(sentence, n):
        tokens = sentence.lower().split()
        return Counter(
            tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)
        )

    score = 0.0
    for n in range(1, 5):
        hyp_ng = get_ngrams(hypothesis, n)
        ref_ng = get_ngrams(reference, n)
        if not hyp_ng or not ref_ng:
            continue
        common = sum((hyp_ng & ref_ng).values())
        denom  = (sum(hyp_ng.values()) + sum(ref_ng.values())) / 2 + 1e-8
        score += common / denom
    return score / 4.0


class SCSTLoss(nn.Module):
    """
    Self-Critical Sequence Training loss (Equations 10–11 in the paper).

        L_R(θ) = - E_{y ~ p_θ}[r(y)]

    Gradient approximation (REINFORCE):
        ∇L_R ≈ -(r(y^s) - r(ŷ)) ∇ log p_θ(y^s)

    where:
      y^s = caption sampled from model (exploration)
      ŷ   = caption from greedy decoding (baseline)
      r   = reward function (CIDEr score)

    The key insight: if the sampled caption scores HIGHER than the
    greedy baseline, increase its probability. Otherwise, decrease it.
    """

    def forward(self, logits_sample, sample_ids,
                greedy_captions, gt_captions, vocab):
        """
        Args:
            logits_sample  : (B, T, vocab_size) — log probs of sampled seqs
            sample_ids     : (B, T)             — sampled token indices
            greedy_captions: list[list[int]]    — greedy decoded ids
            gt_captions    : list[str]          — ground truth strings
            vocab          : Vocabulary

        Returns:
            scalar SCST loss
        """
        B = len(gt_captions)

        # Decode sampled captions to strings
        sampled_strings = [
            vocab.decode(sample_ids[b].tolist()) for b in range(B)
        ]
        greedy_strings = [
            vocab.decode(greedy_captions[b]) for b in range(B)
        ]

        # Compute rewards: r(y^s) and r(ŷ)
        rewards_sample = torch.tensor(
            [simple_cider_score(sampled_strings[b], gt_captions[b])
             for b in range(B)],
            dtype=torch.float32, device=DEVICE
        )
        rewards_greedy = torch.tensor(
            [simple_cider_score(greedy_strings[b], gt_captions[b])
             for b in range(B)],
            dtype=torch.float32, device=DEVICE
        )

        # Advantage: positive = sampled is better than greedy baseline
        advantage = rewards_sample - rewards_greedy    # (B,)

        # REINFORCE loss: - advantage * log p(y^s)
        log_probs = F.log_softmax(logits_sample, dim=2)  # (B, T, V)
        # Gather log probs of sampled tokens
        T = sample_ids.size(1)
        selected_log_probs = log_probs.gather(
            2, sample_ids.unsqueeze(2)
        ).squeeze(2)                                    # (B, T)
        seq_log_probs = selected_log_probs.mean(dim=1)  # (B,)

        loss = -(advantage * seq_log_probs).mean()
        return loss, rewards_sample.mean().item()


# ============================================================
# SECTION 4: TRAINING
# ============================================================

class Trainer:
    """
    Manages the two-phase training procedure:
      Phase 1 — Cross-entropy (teacher forcing, fast convergence)
      Phase 2 — SCST / RL fine-tuning (optimize CIDEr directly)
    """

    def __init__(self, model, train_loader, val_loader,
                 vocab, cfg):
        self.model        = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.vocab        = vocab
        self.cfg          = cfg

        self.xe_loss  = CrossEntropyLoss(pad_idx=vocab.PAD)
        self.rl_loss  = SCSTLoss()

        self.history = {
            'train_xe'  : [], 'val_xe'   : [],
            'train_rl'  : [], 'val_reward': []
        }

    # ── Phase 1: Cross-Entropy Training ─────────────────────

    def train_xe_epoch(self, optimizer):
        self.model.train()
        total_loss, n = 0.0, 0

        for regions, captions, lengths, _ in self.train_loader:
            regions  = regions.to(DEVICE)
            captions = captions.to(DEVICE)

            optimizer.zero_grad()
            logits, _ = self.model(regions, captions, lengths)

            loss = self.xe_loss(logits, captions, lengths)
            loss.backward()
            nn.utils.clip_grad_norm_(
                self.model.parameters(), self.cfg['grad_clip']
            )
            optimizer.step()

            total_loss += loss.item() * regions.size(0)
            n          += regions.size(0)

        return total_loss / n

    @torch.no_grad()
    def validate_xe(self):
        self.model.eval()
        total_loss, n = 0.0, 0

        for regions, captions, lengths, _ in self.val_loader:
            regions  = regions.to(DEVICE)
            captions = captions.to(DEVICE)
            logits, _ = self.model(regions, captions, lengths)
            loss = self.xe_loss(logits, captions, lengths)
            total_loss += loss.item() * regions.size(0)
            n          += regions.size(0)

        return total_loss / n

    def run_xe_phase(self):
        print("\n" + "="*60)
        print("PHASE 1: Cross-Entropy Training")
        print("="*60)
        optimizer = optim.Adam(
            self.model.parameters(), lr=self.cfg['lr_xe']
        )
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=2, gamma=0.5
        )
        best_val  = float('inf')

        for epoch in range(1, self.cfg['num_epochs_xe'] + 1):
            t0       = time.time()
            tr_loss  = self.train_xe_epoch(optimizer)
            val_loss = self.validate_xe()
            scheduler.step()

            self.history['train_xe'].append(tr_loss)
            self.history['val_xe'].append(val_loss)

            flag = ''
            if val_loss < best_val:
                best_val = val_loss
                torch.save(self.model.state_dict(),
                           OUT / 'best_xe_model.pt')
                flag = ' ← saved'

            print(f"  Epoch {epoch:2d}/{self.cfg['num_epochs_xe']} | "
                  f"Train XE: {tr_loss:.4f} | "
                  f"Val XE: {val_loss:.4f} | "
                  f"Time: {time.time()-t0:.1f}s{flag}")

        # Load best weights before RL phase
        self.model.load_state_dict(
            torch.load(OUT / 'best_xe_model.pt', map_location=DEVICE)
        )
        print(f"\nBest Val XE Loss: {best_val:.4f}")

    # ── Phase 2: SCST / RL Fine-Tuning ──────────────────────

    def train_rl_epoch(self, optimizer, gt_caption_map):
        self.model.train()
        total_loss, total_reward, n = 0.0, 0.0, 0

        for regions, captions, lengths, labels in self.train_loader:
            regions  = regions.to(DEVICE)
            B        = regions.size(0)

            optimizer.zero_grad()

            # ── Greedy baseline ───────────────────────────────
            self.model.eval()
            with torch.no_grad():
                greedy_ids, _ = self.model.generate_greedy(
                    regions, self.vocab
                )
            self.model.train()

            # ── Sample from policy ────────────────────────────
            # Simplified: use greedy with temperature-scaled logits
            v_bar   = self.model._mean_pool(regions)
            h1, c1  = self.model._init_hidden(B)
            h2, c2  = self.model._init_hidden(B)

            word = torch.full((B,), self.vocab.START,
                              dtype=torch.long, device=DEVICE)
            sample_ids   = []
            sample_logits = []
            T_sample = 15

            for _ in range(T_sample):
                w_emb  = self.model.embedding(word)
                x1     = torch.cat([h2, v_bar, w_emb], dim=1)
                h1, c1 = self.model.attention_lstm(x1, (h1, c1))
                v_hat, _ = self.model.attention(regions, h1)
                x2     = torch.cat([v_hat, h1], dim=1)
                h2, c2 = self.model.language_lstm(x2, (h2, c2))
                logit  = self.model.fc(h2)               # (B, V)
                # Sample with temperature
                probs  = F.softmax(logit / 1.2, dim=1)
                word   = torch.multinomial(probs, 1).squeeze(1)
                sample_ids.append(word)
                sample_logits.append(logit)

            sample_ids_t    = torch.stack(sample_ids,    dim=1)  # (B,T)
            sample_logits_t = torch.stack(sample_logits, dim=1)  # (B,T,V)

            # Ground-truth captions
            gt_strings = [
                CAPTION_TEMPLATES[CIFAR10_LABELS[labels[b].item()]][0]
                for b in range(B)
            ]

            loss, reward = self.rl_loss(
                sample_logits_t, sample_ids_t,
                greedy_ids, gt_strings, self.vocab
            )

            loss.backward()
            nn.utils.clip_grad_norm_(
                self.model.parameters(), self.cfg['grad_clip']
            )
            optimizer.step()

            total_loss   += loss.item()   * B
            total_reward += reward        * B
            n            += B

        return total_loss / n, total_reward / n

    def run_rl_phase(self):
        print("\n" + "="*60)
        print("PHASE 2: SCST / Reinforcement Learning Fine-Tuning")
        print("="*60)
        optimizer = optim.Adam(
            self.model.parameters(), lr=self.cfg['lr_rl']
        )
        gt_caption_map = {}  # placeholder (populated per batch above)
        best_reward    = -float('inf')

        for epoch in range(1, self.cfg['num_epochs_rl'] + 1):
            t0   = time.time()
            loss, reward = self.train_rl_epoch(optimizer, gt_caption_map)

            self.history['train_rl'].append(loss)
            self.history['val_reward'].append(reward)

            flag = ''
            if reward > best_reward:
                best_reward = reward
                torch.save(self.model.state_dict(),
                           OUT / 'best_rl_model.pt')
                flag = ' ← saved'

            print(f"  Epoch {epoch:2d}/{self.cfg['num_epochs_rl']} | "
                  f"RL Loss: {loss:.4f} | "
                  f"Avg Reward: {reward:.4f} | "
                  f"Time: {time.time()-t0:.1f}s{flag}")

        # Load best RL weights
        self.model.load_state_dict(
            torch.load(OUT / 'best_rl_model.pt', map_location=DEVICE)
        )
        print(f"\nBest Avg Reward: {best_reward:.4f}")

    def run(self):
        self.run_xe_phase()
        self.run_rl_phase()
        return self.history


# ============================================================
# SECTION 5: EVALUATION METRICS
# ============================================================

def compute_bleu(hypothesis, reference, max_n=4):
    """
    BLEU-N: Bilingual Evaluation Understudy.

    Measures n-gram precision of hypothesis against reference,
    with brevity penalty for short outputs.

    Higher is better. Range: [0, 1] (reported as 0–100).
    """
    from collections import Counter

    def ngrams(tokens, n):
        return Counter(
            tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)
        )

    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()

    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref_tokens) / (len(hyp_tokens) + 1e-8)))

    score = 0.0
    for n in range(1, max_n + 1):
        hyp_ng = ngrams(hyp_tokens, n)
        ref_ng = ngrams(ref_tokens, n)
        common = sum((hyp_ng & ref_ng).values())
        total  = sum(hyp_ng.values()) + 1e-8
        score += math.log(common / total + 1e-8)

    bleu = bp * math.exp(score / max_n)
    return bleu


def compute_meteor(hypothesis, reference):
    """
    METEOR: Metric for Evaluation of Translation with Explicit ORdering.

    Computes unigram F-score with harmonic mean, then applies a
    chunk-based fragmentation penalty for word ordering.

    Higher is better.
    """
    hyp = set(hypothesis.lower().split())
    ref = set(reference.lower().split())
    if not hyp or not ref:
        return 0.0

    matches = len(hyp & ref)
    if matches == 0:
        return 0.0

    precision = matches / len(hyp)
    recall    = matches / len(ref)
    F_mean    = (10 * precision * recall) / (9 * precision + recall + 1e-8)
    penalty   = 0.5 * ((matches / (len(hyp) + 1e-8)) ** 3)
    meteor    = F_mean * (1 - penalty)
    return meteor


def compute_rouge_l(hypothesis, reference):
    """
    ROUGE-L: Recall-Oriented Understudy for Gisting Evaluation.

    Based on the length of the Longest Common Subsequence (LCS).
    Measures fluency and coverage simultaneously.

    Higher is better.
    """
    hyp = hypothesis.lower().split()
    ref = reference.lower().split()
    m, n = len(hyp), len(ref)
    if m == 0 or n == 0:
        return 0.0

    # Dynamic programming LCS
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if hyp[i - 1] == ref[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs       = dp[m][n]
    precision = lcs / (m + 1e-8)
    recall    = lcs / (n + 1e-8)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_cider_simple(hypothesis, reference):
    """
    Simplified CIDEr score (n-gram consensus similarity).
    Used as the RL reward and for ranking.
    Higher is better.
    """
    return simple_cider_score(hypothesis, reference)


def evaluate_model(model, loader, vocab, num_samples=50):
    """
    Evaluate the model on a DataLoader subset.
    Returns average BLEU-4, METEOR, ROUGE-L, CIDEr.
    """
    model.eval()
    bleu_scores, meteor_scores = [], []
    rouge_scores, cider_scores = [], []

    count = 0
    with torch.no_grad():
        for regions, captions, lengths, labels in loader:
            if count >= num_samples:
                break
            regions = regions.to(DEVICE)
            B       = regions.size(0)

            pred_ids, _ = model.generate_greedy(regions, vocab)

            for b in range(B):
                lbl   = labels[b].item()
                gt    = CAPTION_TEMPLATES[CIFAR10_LABELS[lbl]][0]
                pred  = vocab.decode(pred_ids[b])

                bleu_scores.append(compute_bleu(pred, gt))
                meteor_scores.append(compute_meteor(pred, gt))
                rouge_scores.append(compute_rouge_l(pred, gt))
                cider_scores.append(compute_cider_simple(pred, gt))

            count += B

    metrics = {
        'BLEU-4'  : np.mean(bleu_scores)   * 100,
        'METEOR'  : np.mean(meteor_scores)  * 100,
        'ROUGE-L' : np.mean(rouge_scores)   * 100,
        'CIDEr'   : np.mean(cider_scores)   * 100,
    }
    return metrics


# ============================================================
# SECTION 6: VISUALIZATION — 8-PANEL DASHBOARD
# ============================================================

def create_dashboard(model, val_ds, vocab, history, metrics):
    """
    8-Panel Publication-Quality Dashboard.

    Panel A — Training / Validation Loss curves (XE phase)
    Panel B — SCST Reward progression (RL phase)
    Panel C — Attention heatmap: word-by-word caption generation
    Panel D — Region attention distribution bar chart
    Panel E — Evaluation metrics bar chart (BLEU/METEOR/ROUGE/CIDEr)
    Panel F — Sample generated vs ground-truth caption comparison
    Panel G — Attention weight matrix over caption time steps
    Panel H — Architecture schematic diagram
    """

    fig = plt.figure(figsize=(22, 28))
    fig.patch.set_facecolor('white')
    gs  = gridspec.GridSpec(4, 2, figure=fig,
                            hspace=0.45, wspace=0.35)

    model.eval()
    TITLE_FONT  = {'fontsize': 13, 'fontweight': 'bold', 'color': 'black'}
    LABEL_FONT  = {'fontsize': 10, 'color': 'black'}
    TICK_FONT   = {'labelsize': 9}

    # ── Collect one inference example ────────────────────────
    sample_idx   = random.randint(0, len(val_ds) - 1)
    regions_s, captions_s, length_s, label_s = val_ds[sample_idx]
    regions_s = regions_s.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_ids, pred_alphas = model.generate_greedy(
            regions_s, vocab, max_len=12
        )

    pred_str = vocab.decode(pred_ids[0])
    gt_str   = CAPTION_TEMPLATES[CIFAR10_LABELS[label_s]][0]
    words    = pred_str.split()
    T_vis    = min(len(words), 8)
    k_vis    = CFG['num_regions']

    # ── Panel A: XE Loss Curves ───────────────────────────────
    ax_a = fig.add_subplot(gs[0, 0])
    xe_epochs = range(1, len(history['train_xe']) + 1)
    ax_a.plot(xe_epochs, history['train_xe'], 'b-o',
              linewidth=2, markersize=5, label='Train XE Loss')
    ax_a.plot(xe_epochs, history['val_xe'],   'r-s',
              linewidth=2, markersize=5, label='Val XE Loss')
    ax_a.set_xlabel('Epoch', **LABEL_FONT)
    ax_a.set_ylabel('Cross-Entropy Loss', **LABEL_FONT)
    ax_a.set_title('Panel A: Cross-Entropy Training Curves',
                   **TITLE_FONT)
    ax_a.legend(fontsize=9)
    ax_a.tick_params(**TICK_FONT)
    ax_a.set_facecolor('white')
    ax_a.grid(True, alpha=0.3)

    # ── Panel B: SCST Reward ──────────────────────────────────
    ax_b = fig.add_subplot(gs[0, 1])
    rl_epochs = range(1, len(history['train_rl']) + 1)
    ax_b.plot(rl_epochs, history['train_rl'], 'g-^',
              linewidth=2, markersize=6, label='RL Loss')
    ax_b.plot(rl_epochs, history['val_reward'], 'm-d',
              linewidth=2, markersize=6, label='CIDEr Reward')
    ax_b.set_xlabel('Epoch', **LABEL_FONT)
    ax_b.set_ylabel('Value', **LABEL_FONT)
    ax_b.set_title('Panel B: SCST Reward Progression (RL Phase)',
                   **TITLE_FONT)
    ax_b.legend(fontsize=9)
    ax_b.tick_params(**TICK_FONT)
    ax_b.set_facecolor('white')
    ax_b.grid(True, alpha=0.3)

    # ── Panel C: Attention Heatmap ────────────────────────────
    ax_c = fig.add_subplot(gs[1, 0])
    if T_vis > 0 and pred_alphas.size(1) >= T_vis:
        attn_matrix = pred_alphas[0, :T_vis, :k_vis].cpu().numpy()
        # Show top-10 regions for clarity
        top_k = min(10, k_vis)
        top_idxs = attn_matrix.mean(axis=0).argsort()[::-1][:top_k]
        attn_sub  = attn_matrix[:, top_idxs]
        im = ax_c.imshow(attn_sub, cmap='YlOrRd', aspect='auto',
                         vmin=0, vmax=attn_sub.max())
        ax_c.set_xticks(range(top_k))
        ax_c.set_xticklabels([f'R{i}' for i in top_idxs],
                              fontsize=7, rotation=45)
        ax_c.set_yticks(range(T_vis))
        ax_c.set_yticklabels(words[:T_vis], fontsize=8)
        ax_c.set_xlabel('Top-10 Image Regions', **LABEL_FONT)
        ax_c.set_ylabel('Generated Words', **LABEL_FONT)
        ax_c.set_title('Panel C: Attention Heatmap\n'
                       '(word × attended region)', **TITLE_FONT)
        plt.colorbar(im, ax=ax_c, fraction=0.046, pad=0.04)
    else:
        ax_c.text(0.5, 0.5, 'Insufficient caption length',
                  ha='center', va='center', transform=ax_c.transAxes)
        ax_c.set_title('Panel C: Attention Heatmap', **TITLE_FONT)

    # ── Panel D: Region Attention Distribution ────────────────
    ax_d = fig.add_subplot(gs[1, 1])
    if T_vis > 0 and pred_alphas.size(1) >= 1:
        mean_alpha = pred_alphas[0, :T_vis, :].mean(dim=0).cpu().numpy()
        top_20_idx = mean_alpha.argsort()[::-1][:20]
        top_20_val = mean_alpha[top_20_idx]
        colors     = cm.viridis(top_20_val / (top_20_val.max() + 1e-8))
        ax_d.bar(range(20), top_20_val, color=colors, edgecolor='black',
                 linewidth=0.5)
        ax_d.set_xticks(range(20))
        ax_d.set_xticklabels([f'R{i}' for i in top_20_idx],
                              fontsize=7, rotation=45)
        ax_d.set_xlabel('Image Region Index', **LABEL_FONT)
        ax_d.set_ylabel('Mean Attention Weight', **LABEL_FONT)
        ax_d.set_title('Panel D: Mean Attention over Regions\n'
                       '(top 20 most attended)', **TITLE_FONT)
    ax_d.tick_params(**TICK_FONT)
    ax_d.set_facecolor('white')
    ax_d.grid(True, alpha=0.3, axis='y')

    # ── Panel E: Evaluation Metrics ───────────────────────────
    ax_e = fig.add_subplot(gs[2, 0])
    metric_names = list(metrics.keys())
    metric_vals  = list(metrics.values())
    colors_e     = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
    bars = ax_e.bar(metric_names, metric_vals,
                    color=colors_e, edgecolor='black',
                    linewidth=0.8, width=0.5)
    for bar, val in zip(bars, metric_vals):
        ax_e.text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.3,
                  f'{val:.1f}', ha='center', va='bottom',
                  fontsize=10, fontweight='bold')
    ax_e.set_ylabel('Score', **LABEL_FONT)
    ax_e.set_title('Panel E: Evaluation Metrics Summary',
                   **TITLE_FONT)
    ax_e.set_ylim(0, max(metric_vals) * 1.25)
    ax_e.tick_params(**TICK_FONT)
    ax_e.set_facecolor('white')
    ax_e.grid(True, alpha=0.3, axis='y')

    # ── Panel F: Generated vs Ground-Truth Captions ───────────
    ax_f = fig.add_subplot(gs[2, 1])
    ax_f.axis('off')
    ax_f.set_facecolor('#F8F9FA')
    ax_f.set_title('Panel F: Caption Comparison', **TITLE_FONT)

    info_text = (
        f"Class Label : {CIFAR10_LABELS[label_s].upper()}\n\n"
        f"Ground Truth:\n  \"{gt_str}\"\n\n"
        f"Generated (Greedy):\n  \"{pred_str}\"\n\n"
        f"BLEU-4  : {compute_bleu(pred_str, gt_str)*100:.1f}\n"
        f"METEOR  : {compute_meteor(pred_str, gt_str)*100:.1f}\n"
        f"ROUGE-L : {compute_rouge_l(pred_str, gt_str)*100:.1f}\n"
        f"CIDEr   : "
        f"{compute_cider_simple(pred_str, gt_str)*100:.1f}"
    )
    ax_f.text(0.05, 0.95, info_text,
              transform=ax_f.transAxes,
              fontsize=10, verticalalignment='top',
              fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='#E3F2FD',
                        edgecolor='#1565C0', linewidth=1.5))

    # ── Panel G: Attention Weight Matrix ──────────────────────
    ax_g = fig.add_subplot(gs[3, 0])
    if T_vis > 1 and pred_alphas.size(1) >= T_vis:
        attn_full = pred_alphas[0, :T_vis, :].cpu().numpy()
        im_g = ax_g.imshow(attn_full, cmap='Blues',
                           aspect='auto', interpolation='nearest')
        ax_g.set_xlabel('Region Index (0–35)', **LABEL_FONT)
        ax_g.set_ylabel('Time Step', **LABEL_FONT)
        ax_g.set_title('Panel G: Full Attention Weight Matrix\n'
                       '(all time steps × all regions)', **TITLE_FONT)
        ax_g.set_yticks(range(T_vis))
        ax_g.set_yticklabels(words[:T_vis], fontsize=8)
        plt.colorbar(im_g, ax=ax_g, fraction=0.046, pad=0.04)
    else:
        ax_g.text(0.5, 0.5, 'Insufficient data for matrix',
                  ha='center', va='center', transform=ax_g.transAxes)
        ax_g.set_title('Panel G: Attention Weight Matrix', **TITLE_FONT)

    # ── Panel H: Architecture Schematic ───────────────────────
    ax_h = fig.add_subplot(gs[3, 1])
    ax_h.axis('off')
    ax_h.set_facecolor('white')
    ax_h.set_title('Panel H: Up-Down Architecture Overview',
                   **TITLE_FONT)

    boxes = [
        (0.10, 0.82, '#BBDEFB', 'INPUT IMAGE\n(H × W × 3)'),
        (0.10, 0.64, '#C8E6C9', 'FASTER R-CNN\n(Bottom-Up)'),
        (0.10, 0.46, '#FFF9C4', 'REGION FEATURES\nV = {v₁,...,v₃₆}\n∈ ℝ²⁰⁴⁸'),
        (0.55, 0.82, '#F8BBD0', 'WORD EMBEDDING\nWₑ Πₜ'),
        (0.55, 0.64, '#E1BEE7', 'TOP-DOWN LSTM\nh¹ₜ = LSTM([h²ₜ₋₁, v̄, wₑ])'),
        (0.55, 0.46, '#B2EBF2', 'SOFT ATTENTION\nv̂ₜ = Σ αᵢ vᵢ'),
        (0.55, 0.28, '#DCEDC8', 'LANGUAGE LSTM\nh²ₜ = LSTM([v̂ₜ, h¹ₜ])'),
        (0.55, 0.10, '#FFE0B2', 'SOFTMAX OUTPUT\np(yₜ) = softmax(Wₚh²ₜ)'),
    ]

    for (x, y, color, text) in boxes:
        fancy = patches.FancyBboxPatch(
            (x, y), 0.35, 0.14,
            boxstyle='round,pad=0.01',
            facecolor=color, edgecolor='#555',
            linewidth=1.2, transform=ax_h.transAxes
        )
        ax_h.add_patch(fancy)
        ax_h.text(x + 0.175, y + 0.07, text,
                  ha='center', va='center',
                  fontsize=7.5, fontweight='bold',
                  transform=ax_h.transAxes)

    # Draw arrows between boxes
    arrow_pairs = [
        (0.275, 0.82, 0.275, 0.78),   # Image → Faster R-CNN
        (0.275, 0.64, 0.275, 0.60),   # R-CNN → Regions
        (0.45,  0.53, 0.55, 0.53),    # Regions → Attention
        (0.725, 0.82, 0.725, 0.78),   # Embedding → TD-LSTM
        (0.725, 0.64, 0.725, 0.60),   # TD-LSTM → Attention
        (0.725, 0.46, 0.725, 0.42),   # Attention → Lang-LSTM
        (0.725, 0.28, 0.725, 0.24),   # Lang-LSTM → Softmax
    ]
    for (x1, y1, x2, y2) in arrow_pairs:
        ax_h.annotate('', xy=(x2, y2), xytext=(x1, y1),
                      xycoords='axes fraction',
                      textcoords='axes fraction',
                      arrowprops=dict(arrowstyle='->', color='#333',
                                     lw=1.5))

    # Main figure title
    fig.suptitle(
        'Bottom-Up and Top-Down Attention for Image Captioning\n'
        'Anderson et al. (CVPR 2018) — Educational PyTorch Implementation',
        fontsize=15, fontweight='bold', y=0.98, color='black'
    )

    plt.savefig(OUT / 'updown_dashboard.png',
                dpi=150, bbox_inches='tight',
                facecolor='white')
    plt.close()
    print(f"\nDashboard saved → {OUT / 'updown_dashboard.png'}")


# ============================================================
# SECTION 7: VQA MODEL
# ============================================================
# A simplified demonstration of the paper's VQA branch.
# Uses gated tanh activations (Equations 12–14) and
# the same bottom-up region features for multi-label
# classification over candidate answers.
# ============================================================

class GatedTanh(nn.Module):
    """
    Gated Hyperbolic Tangent activation (Equations 12–14).

        ỹ = tanh(W x + b)
        g = σ(W' x + b')
        y = ỹ ⊙ g

    The sigmoid gate g controls how much of the tanh activation
    passes through. Empirically outperforms ReLU/tanh on VQA.
    """

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear      = nn.Linear(in_dim, out_dim)
        self.linear_gate = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        y_tilde = torch.tanh(self.linear(x))
        g       = torch.sigmoid(self.linear_gate(x))
        return y_tilde * g


class VQAModel(nn.Module):
    """
    Up-Down VQA Model (Section 3.3 of the paper).

    Architecture:
      Question → GRU → q                      (question encoding)
      q + V   → soft attention → v̂            (attended image feature)
      h = f_q(q) ⊙ f_v(v̂)                    (joint embedding)
      p(y) = σ(W_o f_o(h))                    (multi-label output)
    """

    def __init__(self, vocab_size, region_dim, embed_dim,
                 hidden_dim, num_answers):
        super().__init__()

        self.vocab_size  = vocab_size
        self.hidden_dim  = hidden_dim
        self.num_answers = num_answers

        # Question encoder: embedding + GRU
        self.q_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru     = nn.GRU(embed_dim, hidden_dim,
                              batch_first=True, num_layers=1)

        # Attention over image regions conditioned on question
        self.attn_W  = nn.Linear(region_dim + hidden_dim, hidden_dim)
        self.attn_w  = nn.Linear(hidden_dim, 1)

        # Gated tanh transformation for question and image
        self.f_q = GatedTanh(hidden_dim, hidden_dim)
        self.f_v = GatedTanh(region_dim, hidden_dim)
        self.f_o = GatedTanh(hidden_dim, hidden_dim)

        # Answer classifier (multi-label → sigmoid)
        self.classifier = nn.Linear(hidden_dim, num_answers)

    def forward(self, V, question_ids):
        """
        Args:
            V            : (B, k, region_dim) — region features
            question_ids : (B, Q)             — encoded question tokens

        Returns:
            logits : (B, num_answers) — raw scores per answer
            alpha  : (B, k)          — attention weights
        """
        B, k, _ = V.size()

        # ── Question Encoding ─────────────────────────────────
        q_emb = self.q_embed(question_ids)      # (B, Q, embed_dim)
        _, q  = self.gru(q_emb)                 # q: (1, B, hidden_dim)
        q     = q.squeeze(0)                    # (B, hidden_dim)

        # ── Question-Conditioned Attention ────────────────────
        # a_i = w_a^T f_a([v_i, q])   (Equation 15)
        q_exp = q.unsqueeze(1).expand(B, k, -1) # (B, k, hidden_dim)
        cat   = torch.cat([V, q_exp], dim=2)    # (B, k, D+H)
        energy = self.attn_w(torch.tanh(self.attn_W(cat))).squeeze(2)
        alpha  = F.softmax(energy, dim=1)       # (B, k)

        # Attended image feature
        v_hat  = (alpha.unsqueeze(2) * V).sum(dim=1)  # (B, region_dim)

        # ── Joint Embedding ───────────────────────────────────
        # h = f_q(q) ⊙ f_v(v̂)   (Equation 16)
        h = self.f_q(q) * self.f_v(v_hat)      # (B, hidden_dim)

        # ── Answer Prediction ─────────────────────────────────
        # p(y) = σ(W_o f_o(h))   (Equation 17)
        logits = self.classifier(self.f_o(h))   # (B, num_answers)

        return logits, alpha


def demo_vqa(vocab):
    """
    Demonstrate the VQA model forward pass with synthetic data.
    """
    print("\n" + "="*60)
    print("VQA MODEL DEMO")
    print("="*60)

    NUM_ANSWERS = 50
    vqa_model   = VQAModel(
        vocab_size  = len(vocab),
        region_dim  = CFG['region_feat_dim'],
        embed_dim   = CFG['embed_dim'],
        hidden_dim  = CFG['lstm_hidden_dim'],
        num_answers = NUM_ANSWERS
    ).to(DEVICE)

    # Synthetic batch
    B         = 4
    V         = torch.randn(B, CFG['num_regions'],
                            CFG['region_feat_dim']).to(DEVICE)
    questions = torch.randint(4, len(vocab), (B, 10)).to(DEVICE)

    with torch.no_grad():
        logits, alphas = vqa_model(V, questions)

    probs    = torch.sigmoid(logits)
    top_ans  = probs.argmax(dim=1)

    print(f"  Input regions  : {V.shape}")
    print(f"  Question tokens: {questions.shape}")
    print(f"  Output logits  : {logits.shape}")
    print(f"  Attention (α)  : {alphas.shape}")
    print(f"  Top answer idx : {top_ans.tolist()}")

    # Quick training step to verify gradients
    optimizer = optim.Adam(vqa_model.parameters(), lr=1e-3)
    targets   = torch.zeros(B, NUM_ANSWERS, device=DEVICE)
    for b in range(B):
        targets[b, random.randint(0, NUM_ANSWERS-1)] = 1.0

    vqa_logits, _ = vqa_model(V, questions)
    vqa_loss      = F.binary_cross_entropy_with_logits(
        vqa_logits, targets
    )
    vqa_loss.backward()
    optimizer.step()

    print(f"  VQA Training loss (1 step): {vqa_loss.item():.4f}")
    print("  VQA model: PASSED\n")

    return vqa_model


# ============================================================
# SECTION 8: MAIN PIPELINE
# ============================================================

def main():
    print("="*60)
    print("UP-DOWN ATTENTION — FULL PYTORCH PIPELINE")
    print("Anderson et al., CVPR 2018")
    print("="*60)

    # ── 1. Vocabulary ────────────────────────────────────────
    print("\nStep 1: Building vocabulary...")
    vocab, _ = build_vocab_and_data()

    # ── 2. Datasets and DataLoaders ──────────────────────────
    print("\nStep 2: Loading datasets...")
    train_ds, val_ds, test_ds = build_datasets(vocab)
    train_loader, val_loader, test_loader = build_loaders(
        train_ds, val_ds, test_ds
    )

    # ── 3. Validate feature shapes ───────────────────────────
    print("\nStep 3: Shape validation...")
    reg, cap, length, lbl = next(iter(train_loader))
    print(f"  Regions  : {reg.shape}    — (B, k, D)")
    print(f"  Captions : {cap.shape}    — (B, max_len)")
    print(f"  Lengths  : {length.shape}  — (B,)")

    # CPU forward-pass sanity check (before GPU)
    print("\nStep 4: CPU forward-pass sanity check...")
    model_cpu = UpDownCaptioner(
        vocab_size    = len(vocab),
        region_dim    = CFG['region_feat_dim'],
        embed_dim     = CFG['embed_dim'],
        lstm_dim      = CFG['lstm_hidden_dim'],
        attention_dim = CFG['attention_dim'],
        dropout       = CFG['dropout']
    )
    with torch.no_grad():
        logits_test, alphas_test = model_cpu(reg, cap, length)
    print(f"  Logits : {logits_test.shape}  — (B, T-1, |Σ|)")
    print(f"  Alphas : {alphas_test.shape}  — (B, T-1, k)")
    print(f"  Attention sum (should ≈ 1.0): "
          f"{alphas_test[0, 0].sum().item():.4f}")
    del model_cpu

    # ── 5. Full model on device ───────────────────────────────
    print(f"\nStep 5: Initializing Up-Down model on {DEVICE}...")
    model = UpDownCaptioner(
        vocab_size    = len(vocab),
        region_dim    = CFG['region_feat_dim'],
        embed_dim     = CFG['embed_dim'],
        lstm_dim      = CFG['lstm_hidden_dim'],
        attention_dim = CFG['attention_dim'],
        dropout       = CFG['dropout']
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    train_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)
    print(f"  Total parameters     : {total_params:,}")
    print(f"  Trainable parameters : {train_params:,}")

    # ── 6. Training ───────────────────────────────────────────
    print("\nStep 6: Training...")
    trainer = Trainer(model, train_loader, val_loader, vocab, CFG)
    history = trainer.run()

    # ── 7. Evaluation ─────────────────────────────────────────
    print("\nStep 7: Evaluating on test set...")
    metrics = evaluate_model(model, test_loader, vocab,
                             num_samples=CFG['num_test_samples'])
    print("\n  ┌──────────────┬────────────┐")
    print("  │ Metric       │ Score      │")
    print("  ├──────────────┼────────────┤")
    for name, val in metrics.items():
        print(f"  │ {name:<12} │ {val:>8.2f}   │")
    print("  └──────────────┴────────────┘")

    # ── 8. Beam Search Inference Demo ─────────────────────────
    print("\nStep 8: Beam search inference examples...")
    model.eval()
    for i in range(3):
        reg_i, _, _, lbl_i = val_ds[i]
        reg_i = reg_i.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            beam_ids = model.generate_beam(
                reg_i, vocab, beam_size=CFG['beam_size']
            )
        pred = vocab.decode(beam_ids)
        gt   = CAPTION_TEMPLATES[CIFAR10_LABELS[lbl_i]][0]
        print(f"\n  Sample {i+1}: [{CIFAR10_LABELS[lbl_i]}]")
        print(f"    GT   : {gt}")
        print(f"    Beam : {pred}")

    # ── 9. VQA Demo ───────────────────────────────────────────
    print("\nStep 9: VQA model demo...")
    demo_vqa(vocab)

    # ── 10. Dashboard ─────────────────────────────────────────
    print("\nStep 10: Creating 8-panel dashboard...")
    create_dashboard(model, val_ds, vocab, history, metrics)

    # ── Final Summary ─────────────────────────────────────────
    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)
    print("\nMathematical Pipeline Summary:")
    print("  1.  Image input     → (B, 3, H, W)")
    print("  2.  Region proposals → Faster R-CNN bounding boxes")
    print("  3.  Feature extract  → V ∈ ℝ^{B×k×2048}")
    print("  4.  Bottom-up attn  → salient object regions")
    print("  5.  v̄ = (1/k)ΣᵢVᵢ  → global image summary")
    print("  6.  h¹ₜ = LSTM([h²ₜ₋₁, v̄, Wₑπₜ])  → attention LSTM")
    print("  7.  αₜ = softmax(wₐᵀ tanh(WᵥₐV + Wₕₐh¹))  → weights")
    print("  8.  v̂ₜ = Σᵢ αᵢ,ₜ vᵢ  → attended feature")
    print("  9.  h²ₜ = LSTM([v̂ₜ, h¹ₜ])  → language LSTM")
    print(" 10.  p(yₜ) = softmax(Wₚh²ₜ)  → word distribution")
    print(" 11.  L_XE = -Σ log p(y*ₜ)   → cross-entropy loss")
    print(" 12.  L_RL ≈ -(r(yˢ)-r(ŷ)) ∇ log p(yˢ)  → SCST")
    print(" 13.  Metrics: BLEU / METEOR / ROUGE-L / CIDEr")
    print(f"\n  Output dashboard → {OUT / 'updown_dashboard.png'}")

# Related Works Reference Table
## Bottom-Up and Top-Down Attention for Image Captioning and Visual Question Answering
### Anderson et al., CVPR 2018

---

| # | Author(s) | Year | Title | Venue | Connection to This Paper |
|---|---|---|---|---|---|
| 1 | Xu, K., Ba, J., Kiros, R., Cho, K., Courville, A., Salakhutdinov, R., Zemel, R. S., & Bengio, Y. | 2015 | Show, Attend and Tell: Neural Image Caption Generation with Visual Attention | ICML | Foundational top-down soft attention model for image captioning; operates over uniform CNN spatial grids — the primary baseline architecture that this paper extends by replacing grid features with object-level region proposals |
| 2 | Lu, J., Xiong, C., Parikh, D., & Socher, R. | 2017 | Knowing When to Look: Adaptive Attention via a Visual Sentinel for Image Captioning | CVPR | Proposes adaptive top-down attention with a visual sentinel for captioning; cited as a representative prior work using CNN spatial features with top-down context, which the proposed method outperforms |
| 3 | Rennie, S. J., Marcheret, E., Mroueh, Y., Ross, J., & Goel, V. | 2017 | Self-Critical Sequence Training for Image Captioning | CVPR | Introduces SCST (Self-Critical Sequence Training), the REINFORCE-based RL optimization method adopted directly in this paper to optimize CIDEr score; serves as the primary training baseline for the captioning experiments |
| 4 | Yang, Z., Yuan, Y., Wu, Y., Salakhutdinov, R., & Cohen, W. W. | 2016 | Review Networks for Caption Generation | NeurIPS | Top-down attention captioning model operating over CNN features; included in the MSCOCO test server comparison table as a prior state-of-the-art reference |
| 5 | Pedersoli, M., Lucas, T., Schmid, C., & Verbeek, J. | 2017 | Areas of Attention for Image Captioning | ICCV | One of only two prior works identified that apply attention to salient image regions rather than uniform grids; uses edge boxes or spatial transformers — directly contrasted with the proposed Faster R-CNN approach |
| 6 | Jin, J., Fu, K., Cui, R., Sha, F., & Zhang, C. | 2015 | Aligning Where to See and What to Tell: Image Caption with Region-Based Attention and Scene Factorization | arXiv | The other prior work applying region-based attention for captioning; uses selective search for region proposals filtered by a classifier — contrasted with the proposed Faster R-CNN pretraining strategy |
| 7 | Fukui, A., Park, D. H., Yang, D., Rohrbach, A., Darrell, T., & Rohrbach, M. | 2016 | Multimodal Compact Bilinear Pooling for Visual Question Answering and Visual Grounding | EMNLP | Representative VQA model using top-down attention over CNN features; cited as a prior state-of-the-art baseline in the VQA test server comparison and as a related top-down approach |
| 8 | Lu, J., Yang, J., Batra, D., & Parikh, D. | 2016 | Hierarchical Question-Image Co-Attention for Visual Question Answering | NeurIPS | Proposes stacked/hierarchical top-down attention for VQA; cited as an example of a more complex attention scheme that the proposed simple one-pass attention is compared against |
| 9 | Yang, Z., He, X., Gao, J., Deng, L., & Smola, A. J. | 2016 | Stacked Attention Networks for Image Question Answering | CVPR | Introduces multi-step stacked attention for VQA using CNN spatial features; cited alongside other top-down VQA models as representative prior work that the proposed approach supersedes |
| 10 | Jabri, A., Joulin, A., & van der Maaten, L. | 2016 | Revisiting Visual Question Answering Baselines | arXiv | Presents strong VQA baseline architectures with joint multimodal embeddings; the proposed VQA model builds upon and improves this joint embedding framework |
| 11 | Kazemi, V., & Elqursh, A. | 2017 | Show, Ask, Attend, and Answer: A Strong Baseline for Visual Question Answering | arXiv | Provides a strong single-model VQA baseline; referenced as a competitive prior approach in the VQA model design discussion |
| 12 | Xu, H., & Saenko, K. | 2016 | Ask, Attend and Answer: Exploring Question-Guided Spatial Attention for Visual Question Answering | ECCV | Applies spatial attention conditioned on question representation for VQA; cited as a representative top-down VQA attention model that motivates the proposed object-level attention approach |
| 13 | Antol, S., Agrawal, A., Lu, J., Mitchell, M., Batra, D., Zitnick, C. L., & Parikh, D. | 2015 | VQA: Visual Question Answering | ICCV | Introduces the original VQA task and dataset; foundational work that defines the problem space and evaluation protocol used throughout the paper |
| 14 | Goyal, Y., Khot, T., Summers-Stay, D., Batra, D., & Parikh, D. | 2017 | Making the V in VQA Matter: Elevating the Role of Image Understanding in Visual Question Answering | CVPR | Introduces the VQA v2.0 dataset used for all VQA experiments; directly motivates the need for stronger visual grounding, which the bottom-up attention mechanism addresses |
| 15 | Ren, S., He, K., Girshick, R., & Sun, J. | 2015 | Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks | NeurIPS | Core building block of the proposed bottom-up attention model; Faster R-CNN is directly adopted as the region proposal and feature extraction mechanism |
| 16 | He, K., Zhang, X., Ren, S., & Sun, J. | 2016 | Deep Residual Learning for Image Recognition | CVPR | ResNet-101 serves as the CNN backbone within Faster R-CNN for extracting region features; also used as the baseline (ResNet) comparison model in captioning and VQA experiments |
| 17 | Krishna, R., Zhu, Y., Groth, O., Johnson, J., et al. | 2016 | Visual Genome: Connecting Language and Vision Using Crowdsourced Dense Image Annotations | arXiv | Provides the pretraining dataset (108K images, 1,600 object classes, 400 attribute classes) for the bottom-up Faster R-CNN attention model and is used for VQA data augmentation |
| 18 | Uijlings, J. R., Van De Sande, K. E., Gevers, T., & Smeulders, A. W. | 2013 | Selective Search for Object Recognition | IJCV | Hand-crafted region proposal method used in Jin et al.; explicitly contrasted with the proposed Faster R-CNN approach to motivate the superiority of learned, detection-supervised proposals |
| 19 | Zitnick, L., & Dollar, P. | 2014 | Edge Boxes: Locating Object Proposals from Edges | ECCV | Hand-crafted region proposal method used in the Areas of Attention model; contrasted with Faster R-CNN to highlight the advantage of detection-pretrained region proposals |
| 20 | Jaderberg, M., Simonyan, K., Zisserman, A., & Kavukcuoglu, K. | 2015 | Spatial Transformer Networks | NeurIPS | Differentiable region proposal mechanism used in prior captioning work; cited as an alternative to Faster R-CNN and contrasted with the proposed pretraining-based approach |
| 21 | Zhu, Y., Groth, O., Bernstein, M., & Fei-Fei, L. | 2016 | Visual7W: Grounded Question Answering in Images | CVPR | Grounded VQA work cited as a related top-down attention approach in the VQA literature that motivates the broader need for visual grounding in question answering |
| 22 | Liu, S., Zhu, Z., Ye, N., Guadarrama, S., & Murphy, K. | 2017 | Improved Image Captioning via Policy Gradient Optimization of SPIDEr | ICCV | Policy gradient captioning model included in the MSCOCO test server leaderboard comparison; represents a concurrent alternative RL-based captioning approach |
| 23 | Yao, T., Pan, Y., Li, Y., Qiu, Z., & Mei, T. | 2017 | Boosting Image Captioning with Attributes | ICCV | Attribute-enhanced captioning model (LSTM-A3) included in the MSCOCO test server comparison; directly preceded by and outperformed by the proposed Up-Down model |
| 24 | Dauphin, Y. N., Fan, A., Auli, M., & Grangier, D. | 2016 | Language Modeling with Gated Convolutional Networks | arXiv | Introduces gated tanh activations adopted in the VQA model; cited as the source of the gating mechanism shown to outperform standard ReLU or tanh in the VQA network |
| 25 | Srivastava, R. K., Greff, K., & Schmidhuber, J. | 2015 | Highway Networks | arXiv | Gated tanh activations are described as a special case of highway networks; cited to contextualize the theoretical basis of the gating mechanism used in the VQA model |
| 26 | Williams, R. J. | 1992 | Simple Statistical Gradient-Following Algorithms for Connectionist Reinforcement Learning | Machine Learning | Original REINFORCE algorithm; the theoretical foundation for the SCST policy gradient estimator used in the CIDEr optimization training phase |
| 27 | Russakovsky, O., Deng, J., Su, H., Krause, J., et al. | 2015 | ImageNet Large Scale Visual Recognition Challenge | IJCV | ImageNet pretraining is used to initialize the Faster R-CNN ResNet-101 backbone; cited to establish the analogy between ImageNet pretraining for vision and the proposed Visual Genome pretraining for region features |
| 28 | Buschman, T. J., & Miller, E. K. | 2007 | Top-Down Versus Bottom-Up Control of Attention in the Prefrontal and Posterior Parietal Cortices | Science | Neuroscience source for the top-down / bottom-up attention terminology adopted throughout the paper; motivates the biological grounding of the proposed dual-attention framework |
| 29 | Corbetta, M., & Shulman, G. L. | 2002 | Control of Goal-Directed and Stimulus-Driven Attention in the Brain | Nature Reviews Neuroscience | Second neuroscience reference supporting the biological motivation for distinguishing task-driven (top-down) and stimulus-driven (bottom-up) attention mechanisms |
| 30 | Treisman, A. M., & Gelade, G. | 1980 | A Feature-Integration Theory of Attention | Cognitive Psychology | Foundational cognitive psychology paper on the feature binding problem; cited to motivate why object-level attention regions are superior — all features of an object are processed together, solving the binding problem |
| 31 | Treisman, A. | 1982 | Perceptual Grouping and Attention in Visual Search for Features and for Objects | Journal of Experimental Psychology | Supports the argument that attention plays a central role in solving the feature binding problem; reinforces the cognitive plausibility of object-aligned attention regions |

---

## Summary of Related Work Clusters

| Cluster | Papers | Role in the Paper |
|---|---|---|
| Top-Down Captioning Models | Xu et al. (2015), Lu et al. (2017), Yang et al. (2016), Rennie et al. (2017) | Primary baselines; define the conventional grid-based attention paradigm being replaced |
| Region-Based Captioning (Prior) | Jin et al. (2015), Pedersoli et al. (2017) | Only two prior works using non-grid attention; directly contrasted to motivate Faster R-CNN approach |
| VQA Models | Antol et al. (2015), Goyal et al. (2017), Fukui et al. (2016), Lu et al. (2016), Yang et al. (2016), Kazemi & Elqursh (2017), Xu & Saenko (2016), Zhu et al. (2016) | Define the VQA task, dataset, and baseline architectures; benchmarks against which the proposed model achieves first place |
| Object Detection Backbone | Ren et al. (2015), He et al. (2016), Russakovsky et al. (2015) | Core technical components: Faster R-CNN, ResNet-101, and ImageNet pretraining |
| Pretraining Data | Krishna et al. (2016) | Visual Genome provides supervision for bottom-up attention pretraining |
| RL / Sequence Optimization | Rennie et al. (2017), Williams (1992) | SCST and REINFORCE — foundations of the CIDEr optimization phase |
| Gating Mechanisms | Dauphin et al. (2016), Srivastava et al. (2015) | Gated tanh activation used in the VQA model |
| Neuroscience / Cognitive Motivation | Buschman & Miller (2007), Corbetta & Shulman (2002), Treisman & Gelade (1980), Treisman (1982) | Biological and cognitive grounding for the top-down / bottom-up attention distinction and the feature binding problem |